# CARE Scenario Analysis — ATE Calculator

Computes Average Treatment Effects from the estimated ordered probit models, with **every scenario
dimension selectable**: event, activity, base severity level, comparison severity level, and response
category. Panel 2 (population segments) is included too, and **both panels are exported for every event × activity
× response** (section 6) — each of the 35 models uses its own specification, so each has its own table.

This notebook does **not** re-estimate anything — it reads the fitted coefficients from
`model_coefficients_all_events.csv` (all 5 events, final models only) and the event data files, so it
runs in seconds. Re-estimation lives in `Data_Preparation_and_Modeling.ipynb`.

**Inputs expected in the working directory**
- `model_coefficients_all_events.csv`
- `event_data/heat_data.csv`, `cold_data.csv`, `powerout_data.csv`, `earthquake_data.csv`, `flooding_data.csv`

## What an ATE is here

For a chosen manipulated variable (severity level, or a demographic):

1. Take the estimation sample of that activity's final model.
2. **Base scenario** — force the variable to its base value for *every* person, leave everything else at
   observed values, compute each person's `P(category)`.
3. **Comparison scenario** — force it to the comparison value, recompute.
4. Average over people → `P_base`, `P_comp`.
5. `Absolute ATE = P_comp − P_base` (percentage points); `Percent ATE = (P_comp − P_base) / P_base`.

Ordered probit: `P(Y=1) = Φ(τ₁ − z)`, `P(Y=j) = Φ(τⱼ − z) − Φ(τⱼ₋₁ − z)`, `P(Y=J) = 1 − Φ(τ_{J−1} − z)`, with `z = Xβ`.

## 1. Setup

In [1]:
import pandas as pd, numpy as np
from scipy.stats import norm
import itertools, warnings
warnings.filterwarnings('ignore')

COEF = pd.read_csv('./model_coefficients_all_events.csv')

EVENTS = {
    'heat':       dict(label='Extreme Heat',  flag='ext_heat',       file='./event_data/heat_data.csv'),
    'cold':       dict(label='Extreme Cold',  flag='ext_cold',       file='./event_data/cold_data.csv'),
    'powerout':   dict(label='Power Outage',  flag='ext_powerout',   file='./event_data/powerout_data.csv'),
    'earthquake': dict(label='Earthquakes',   flag='ext_earthquake', file='./event_data/earthquake_data.csv'),
    'flooding':   dict(label='Flooding',      flag='ext_flooding',   file='./event_data/flooding_data.csv'),
}

# activity -> dv column suffix pattern per event, and whether the model uses the working sub-sample
ACT_DV = {
 'Usual':   ('lkly_normal_business', False), 'Home':    ('stay_home',   False),
 'Car':     ('car_travel',           False), 'WFH':     ('wfh',         True),
 'WFO':     ('commute',              True),  'Dine in': ('indoor_restaurant', False),
 'Pick up': ('takeout_pickup',       False), 'Delivery':('food_delivery',    False),
}
# transit is named differently for heat than for the rest
TRANSIT_DV = {'heat':'public_transit','cold':'transit_use','powerout':'transit_use',
              'earthquake':'transit_use','flooding':'transit_use'}

ACT_LABEL = {'Usual':'Go about business as usual','Home':'Staying at home','Car':'Using a car for traveling',
 'Transit':'Taking public transit','WFH':'Working from home','WFO':'Working from the office',
 'Dine in':'Eating indoors at a restaurant','Pick up':'Picking up takeout','Delivery':'Having food delivered'}
SEV_LABEL = {1:'Not severe at all',2:'Slightly severe',3:'Moderately severe',4:'Very severe',5:'Extremely severe'}
CAT3 = {1:'Do less',2:'About the same',3:'Do more'}
CAT5 = {1:'Very unlikely',2:'Somewhat unlikely',3:'Neutral',4:'Somewhat likely',5:'Very likely'}

_data_cache = {}
def load_event(ev):
    if ev not in _data_cache:
        _data_cache[ev] = pd.read_csv(EVENTS[ev]['file'], low_memory=False)
    return _data_cache[ev]

def dv_column(ev, act):
    suffix = TRANSIT_DV[ev] if act=='Transit' else ACT_DV[act][0]
    return f'ext_{ev}_{suffix}'

def worker_only(act):
    return True if act in ('WFH','WFO') else False

def available_activities(ev):
    return sorted(COEF[COEF.event==ev].activity.unique().tolist())

print('Events:', {e: available_activities(e) for e in EVENTS})

Events: {'heat': ['Car', 'Delivery', 'Dine in', 'Home', 'Pick up', 'Transit', 'Usual', 'WFH', 'WFO'], 'cold': ['Car', 'Delivery', 'Dine in', 'Home', 'Pick up', 'Transit', 'Usual', 'WFH', 'WFO'], 'powerout': ['Car', 'Delivery', 'Dine in', 'Home', 'Pick up', 'Transit', 'Usual'], 'earthquake': ['Car', 'Home', 'Transit', 'Usual', 'WFH'], 'flooding': ['Car', 'Home', 'Transit', 'Usual', 'WFH']}


## 2. Core functions

`get_model` pulls a fitted model; `cat_probs` turns an index into category probabilities;
`ate_severity` and `ate_segment` are the two ATE engines.

In [2]:
def get_model(ev, act):
    """Return (coefs dict, thresholds list, x_vars, estimation sample, dv name)."""
    d = COEF[(COEF.event==ev) & (COEF.activity==act)]
    if d.empty:
        raise ValueError(f'No model for {act} / {ev}')
    coefs = {r.variable: r.value for r in d[d.term=='coef'].itertuples()}
    thr   = [r.value for r in d[d.term=='threshold'].sort_values('variable').itertuples()]
    x_vars = list(coefs)
    dv = dv_column(ev, act)
    sub = load_event(ev)
    if worker_only(act):
        sub = sub[sub['empsta'] < 3]
    sample = sub[x_vars + [dv]].dropna().copy()
    return coefs, thr, x_vars, sample, dv

def cat_probs(z, thr):
    """Ordered probit category probabilities for an array of indices z."""
    z = np.asarray(z, dtype=float)
    cum = np.column_stack([norm.cdf(t - z) for t in thr])
    cum = np.column_stack([np.zeros(len(z)), cum, np.ones(len(z))])
    return np.diff(cum, axis=1)

def ate_severity(ev, act, base_level, comp_level, collapse_usual=True):
    """ATE of moving everyone from base_level to comp_level of past-event severity.
    base_level / comp_level are 1..5 (1 = 'Not severe at all', the omitted reference)."""
    coefs, thr, x_vars, sample, dv = get_model(ev, act)
    sev_vars = [f'{ev}_imp_{l}' for l in [2,3,4,5]]
    nonsev = [v for v in x_vars if v not in sev_vars]
    xb = sum(sample[v].values*coefs[v] for v in nonsev) if nonsev else np.zeros(len(sample))
    add = lambda L: 0.0 if L==1 else coefs.get(f'{ev}_imp_{L}', 0.0)
    Pb = cat_probs(xb + add(base_level), thr).mean(axis=0)
    Pc = cat_probs(xb + add(comp_level), thr).mean(axis=0)
    J = len(thr)+1
    if J==5 and collapse_usual:            # 5-point likelihood -> 3 dashboard buttons
        grp = [[0,1],[2],[3,4]]
        Pb = np.array([Pb[g].sum() for g in grp]); Pc = np.array([Pc[g].sum() for g in grp])
        labs = CAT3
    else:
        labs = CAT5 if J==5 else CAT3
    out = pd.DataFrame({'Response':[labs[j+1] for j in range(len(Pb))],
                        'P_base':Pb, 'P_comp':Pc, 'ATE_abs':Pc-Pb,
                        'ATE_pct':np.where(Pb>0,(Pc-Pb)/Pb*100,np.nan)})
    out.attrs.update(event=ev, activity=act, n=len(sample),
                     base=SEV_LABEL[base_level], comp=SEV_LABEL[comp_level])
    return out

def _summarise(Pb, Pc, thr, collapse_usual=True):
    """Turn full-category probabilities into the output table (5-point Usual -> 3 buttons)."""
    J = len(thr) + 1
    if J == 5 and collapse_usual:
        grp = [[0,1],[2],[3,4]]
        Pb = np.array([Pb[g].sum() for g in grp]); Pc = np.array([Pc[g].sum() for g in grp]); labs = CAT3
    else:
        labs = CAT5 if J == 5 else CAT3
    return pd.DataFrame({'Response':[labs[j+1] for j in range(len(Pb))],
                         'P_base':Pb, 'P_comp':Pc, 'ATE_abs':Pc-Pb,
                         'ATE_pct':np.where(Pb>0, (Pc-Pb)/Pb*100, np.nan)})

def ate_segment(ev, act, base_spec, comp_spec, collapse_usual=True):
    """ATE of moving everyone from base_spec to comp_spec, holding each person's OWN severity fixed.
    base_spec / comp_spec are dicts {variable: forced value}, e.g. {'female':0} vs {'female':1}.
    Variables that are not in THIS model's specification contribute nothing.
    Returns (table, in_model). in_model is True only if at least one variable that actually differs
    between base and comparison is in the model -- e.g. '3+ persons -> 1 person' is False for a model
    that has hhsize2 but not hhsize1. P_base / P_comp are always reported (equal when in_model=False)."""
    coefs, thr, x_vars, sample, dv = get_model(ev, act)
    xb_obs = sum(sample[v].values*coefs[v] for v in x_vars)
    in_model = [v for v in base_spec if v in coefs]
    zero = np.zeros(len(sample))
    shift_b = sum(((base_spec[v]-sample[v].values)*coefs[v] for v in in_model), zero)
    shift_c = sum(((comp_spec[v]-sample[v].values)*coefs[v] for v in in_model), zero)
    Pb = cat_probs(xb_obs+shift_b, thr).mean(axis=0)
    Pc = cat_probs(xb_obs+shift_c, thr).mean(axis=0)
    out = _summarise(Pb, Pc, thr, collapse_usual)
    effective = [v for v in in_model if base_spec[v] != comp_spec.get(v, base_spec[v])]
    out.attrs.update(event=ev, activity=act, n=len(sample), vars_in_model=effective)
    return out, bool(effective)

def ate_continuous(ev, act, var, unit_shift=1.0, collapse_usual=True):
    """ATE of a fixed additive shift on the normalized continuous-construct scale.
    The baseline keeps each person's current score and the treatment sets x_treat = x_base + 1.
    This matches the paper's '1 Unit Increase' definition: it is neither a 1% increase nor a
    model-specific shift based on the estimation sample's empirical standard deviation."""
    coefs, thr, x_vars, sample, dv = get_model(ev, act)
    xb = sum(sample[v].values*coefs[v] for v in x_vars)
    ok = var in coefs
    delta = float(unit_shift) if ok else 0.0
    Pb = cat_probs(xb, thr).mean(axis=0)
    Pc = cat_probs(xb + (coefs[var]*delta if ok else 0.0), thr).mean(axis=0)
    out = _summarise(Pb, Pc, thr, collapse_usual)
    out.attrs.update(event=ev, activity=act, n=len(sample),
                     shift=f'{unit_shift:g} latent-scale unit',
                     vars_in_model=[var] if ok else [])
    return out, ok

print('functions ready')

functions ready


## 3. Panel 1 — pick any event, base level and comparison level

Change the four values in the cell below and re-run it. **Any** ordered pair of levels works
(1→5, 3→2, 4→5, …), not just comparisons against "Not severe at all".

In [3]:
# ------------------------- CHANGE THESE -------------------------
EVENT       = 'heat'          # 'heat' | 'cold' | 'powerout' | 'earthquake' | 'flooding'
BASE_LEVEL  = 3               # 1..5   (1 = Not severe at all)
COMP_LEVEL  = 4               # 1..5
RESPONSE    = 'Do less'       # 'Do less' | 'About the same' | 'Do more'
# ----------------------------------------------------------------

rows=[]
for act in available_activities(EVENT):
    t = ate_severity(EVENT, act, BASE_LEVEL, COMP_LEVEL)
    r = t[t.Response==RESPONSE].iloc[0]
    rows.append(dict(Activity=ACT_LABEL[act], P_base=r.P_base, P_comp=r.P_comp,
                     ATE_abs=r.ATE_abs, ATE_pct=r.ATE_pct, N=t.attrs['n']))
tbl = pd.DataFrame(rows).sort_values('ATE_pct', ascending=False)

print(f"{EVENTS[EVENT]['label']}  |  {SEV_LABEL[BASE_LEVEL]}  ->  {SEV_LABEL[COMP_LEVEL]}  |  response: {RESPONSE}")
print('='*92)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:8.4f}'))

Extreme Heat  |  Moderately severe  ->  Very severe  |  response: Do less
                      Activity   P_base   P_comp  ATE_abs  ATE_pct    N
    Go about business as usual   0.2626   0.3220   0.0594  22.6202 2653
Eating indoors at a restaurant   0.2329   0.2632   0.0303  13.0098 2653
       Working from the office   0.2045   0.2184   0.0140   6.8261 1271
     Using a car for traveling   0.1760   0.1687  -0.0074  -4.1795 2653
         Taking public transit   0.2440   0.2335  -0.0105  -4.3176 2653
            Picking up takeout   0.2213   0.2104  -0.0109  -4.9170 2653
         Having food delivered   0.1430   0.1053  -0.0377 -26.3508 2653
               Staying at home   0.0589   0.0402  -0.0188 -31.8722 2653
             Working from home   0.0794   0.0442  -0.0352 -44.3051 1271


### 3b. One activity, all five levels at once

Useful for spotting non-monotonic severity effects (e.g. heat/WFH peaks at *Very severe* and falls back at *Extremely severe*, because `heat_imp_5` is not significant in that model).

In [4]:
EVENT, ACTIVITY, RESPONSE = 'heat', 'WFH', 'Do more'

rows=[]
for L in [1,2,3,4,5]:
    t = ate_severity(EVENT, ACTIVITY, 1, L)
    r = t[t.Response==RESPONSE].iloc[0]
    rows.append(dict(Level=SEV_LABEL[L], P=r.P_comp,
                     ATE_abs_vs_L1=r.ATE_abs, ATE_pct_vs_L1=r.ATE_pct))
print(f'{EVENTS[EVENT]["label"]} — {ACT_LABEL[ACTIVITY]} — P({RESPONSE}) at each severity level')
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f'{x:8.4f}'))

Extreme Heat — Working from home — P(Do more) at each severity level
            Level        P  ATE_abs_vs_L1  ATE_pct_vs_L1
Not severe at all   0.2385         0.0000         0.0000
  Slightly severe   0.2387         0.0002         0.0833
Moderately severe   0.2464         0.0079         3.2916
      Very severe   0.3474         0.1088        45.6253
 Extremely severe   0.2590         0.0205         8.5778


### 3c. Full matrix — every base × comparison pair for one activity

The diagonal is zero by construction; the matrix is antisymmetric in absolute ATE.

In [5]:
EVENT, ACTIVITY, RESPONSE = 'heat', 'Home', 'Do more'

M = pd.DataFrame(index=[SEV_LABEL[b] for b in range(1,6)],
                 columns=[SEV_LABEL[c] for c in range(1,6)], dtype=float)
for b in range(1,6):
    for c in range(1,6):
        if b==c:
            M.iloc[b-1,c-1]=0.0; continue
        t=ate_severity(EVENT,ACTIVITY,b,c)
        M.iloc[b-1,c-1]=t[t.Response==RESPONSE].iloc[0].ATE_pct
print(f'Percent ATE (%) — {EVENTS[EVENT]["label"]}, {ACT_LABEL[ACTIVITY]}, P({RESPONSE})')
print('rows = base level, columns = comparison level')
print(M.to_string(float_format=lambda x: f'{x:+7.2f}'))

Percent ATE (%) — Extreme Heat, Staying at home, P(Do more)
rows = base level, columns = comparison level
                   Not severe at all  Slightly severe  Moderately severe  Very severe  Extremely severe
Not severe at all              +0.00           +30.43             +32.62       +54.49            +43.75
Slightly severe               -23.33            +0.00              +1.68       +18.45            +10.21
Moderately severe             -24.60            -1.65              +0.00       +16.49             +8.39
Very severe                   -35.27           -15.57             -14.16        +0.00             -6.95
Extremely severe              -30.44            -9.27              -7.74        +7.47             +0.00


## 4. Optional interactive widgets

If `ipywidgets` is installed, this gives dropdowns instead of editing code. If it isn't, the cell
prints a note and you can keep using the cells above — nothing else depends on it.

In [6]:
try:
    import ipywidgets as W
    from IPython.display import display, clear_output

    w_ev   = W.Dropdown(options=[(v['label'],k) for k,v in EVENTS.items()], value='heat', description='Event:')
    w_base = W.Dropdown(options=[(SEV_LABEL[i],i) for i in range(1,6)], value=1, description='Base:')
    w_comp = W.Dropdown(options=[(SEV_LABEL[i],i) for i in range(1,6)], value=5, description='Comparison:')
    w_resp = W.Dropdown(options=['Do less','About the same','Do more'], value='Do more', description='Response:')
    out = W.Output()

    def refresh(_=None):
        with out:
            clear_output()
            rows=[]
            for act in available_activities(w_ev.value):
                t=ate_severity(w_ev.value, act, w_base.value, w_comp.value)
                r=t[t.Response==w_resp.value].iloc[0]
                rows.append(dict(Activity=ACT_LABEL[act], P_base=r.P_base, P_comp=r.P_comp,
                                 ATE_abs=r.ATE_abs, ATE_pct=r.ATE_pct, N=t.attrs['n']))
            df_=pd.DataFrame(rows).sort_values('ATE_pct',ascending=False)
            print(f'{EVENTS[w_ev.value]["label"]} | {SEV_LABEL[w_base.value]} -> {SEV_LABEL[w_comp.value]} | {w_resp.value}')
            print('='*92)
            print(df_.to_string(index=False, float_format=lambda x: f'{x:8.4f}'))

    for w in (w_ev,w_base,w_comp,w_resp): w.observe(refresh,names='value')
    display(W.VBox([W.HBox([w_ev,w_base]), W.HBox([w_comp,w_resp]), out]))
    refresh()
except ImportError:
    print('ipywidgets not installed - use the editable cells in section 3 instead.')
    print('(pip install ipywidgets)')

## 5. Panel 2 (Panel B) — population segments

Severity is **not** reset here: each person keeps their own observed severity, and only the segment
variable is manipulated.

**Every event × activity has its own model specification**, so every one of the 35 models has its own
Panel B table, and the three responses (*Do less / About the same / Do more*) differ too. A contrast
whose variable is not in that model's specification has an ATE of exactly 0 — flagged as
`in_model = False` (the flag is per contrast: e.g. *3+ persons → 1 person* is `False` for a model that
has `hhsize2` but not `hhsize1`).

`EXTRA_SEGMENTS` covers variables that appear in the final models but are not in the dashboard's
original segment list (employment, education, race, Hispanic, telecommute, A/C, other built-environment
measures). **Check the labels for `edu_bs`, `bs_grad`, `tcom_no` and `ac_c` against the codebook** —
they are written as `variable = 0 / 1` because their wording isn't defined in this notebook. Set
`INCLUDE_EXTRA = False` to export only the original list.

For the continuous constructs (CR, SE, and PR), the base scenario retains each person's current
score and the treatment scenario adds exactly 1.0 on the normalized construct scale. This matches
the paper's *1 Unit Increase* definition; it is not a 1% change and is not rescaled using each
model-specific sample standard deviation. `ATE_abs` is the absolute probability difference, while
`ATE_pct` remains the relative percentage change calculated from the baseline probability.

The cell below shows one model with all three responses side by side; section 6b exports all of them.

In [7]:
SEGMENTS = {
 'Gender':              ('Male', {'female':0}, [('Female', {'female':1})]),
 'Age Group':           ('18-30', {'age_3150':0,'age_5165':0,'age_65p':0},
                          [('31-50',{'age_3150':1,'age_5165':0,'age_65p':0}),
                           ('51-65',{'age_3150':0,'age_5165':1,'age_65p':0}),
                           ('65+',  {'age_3150':0,'age_5165':0,'age_65p':1})]),
 'Household Income':    ('Less than $50k', {'in50':1,'in50100':0},
                          [('$50k-$100k',{'in50':0,'in50100':1}), ('$100k or higher',{'in50':0,'in50100':0})]),
 'Household Size':      ('3+ persons', {'hhsize1':0,'hhsize2':0},
                          [('1 person',{'hhsize1':1,'hhsize2':0}), ('2 persons',{'hhsize1':0,'hhsize2':1})]),
 'Child in household':  ('No child', {'child':0}, [('Has child',{'child':1})]),
 'Disability':          ('No disability', {'dis_yes':0}, [('Has disability',{'dis_yes':1})]),
 'Works outdoors':      ('No', {'work_out':0}, [('Yes',{'work_out':1})]),
 'Zero-vehicle HH':     ('Has vehicle', {'hhveh0':0}, [('Zero vehicle',{'hhveh0':1})]),
 'Stand-alone house':   ('Not stand-alone', {'sa_home':0}, [('Stand-alone house',{'sa_home':1})]),
 'Rural':               ('Not rural', {'rural':0}, [('Rural',{'rural':1})]),
 'Population Density':  ('Low', {'PopDens_medium':0,'PopDens_high':0},
                          [('Medium',{'PopDens_medium':1,'PopDens_high':0}),
                           ('High',  {'PopDens_medium':0,'PopDens_high':1})]),
 'Transit Access':      ('Low', {'TransitAccess_medium':0,'TransitAccess_high':0},
                          [('Medium',{'TransitAccess_medium':1,'TransitAccess_high':0}),
                           ('High',  {'TransitAccess_medium':0,'TransitAccess_high':1})]),
 'Walkability Index':   ('Very low', {'NatWalkInd_low':0,'NatWalkInd_high':0,'NatWalkInd_very_high':0},
                          [('Low',      {'NatWalkInd_low':1,'NatWalkInd_high':0,'NatWalkInd_very_high':0}),
                           ('High',     {'NatWalkInd_low':0,'NatWalkInd_high':1,'NatWalkInd_very_high':0}),
                           ('Very high',{'NatWalkInd_low':0,'NatWalkInd_high':0,'NatWalkInd_very_high':1})]),
}

# Variables used in the final models but missing from the list above. Labels marked '= 0 / = 1' need
# checking against Codebook.xlsx before they go on the dashboard.
EXTRA_SEGMENTS = {
 'Employment status':   ('Worker', {'non_wrkr':0}, [('Non-worker',{'non_wrkr':1})]),
 'Race':                ('Other race', {'white':0,'black':0,'asian':0},
                          [('White',{'white':1,'black':0,'asian':0}),
                           ('Black',{'white':0,'black':1,'asian':0}),
                           ('Asian',{'white':0,'black':0,'asian':1})]),
 'Hispanic':            ('Not Hispanic', {'hispanic_c':0}, [('Hispanic',{'hispanic_c':1})]),
 'Education (edu_bs)':  ('edu_bs = 0', {'edu_bs':0}, [('edu_bs = 1',{'edu_bs':1})]),
 'Education (bs_grad)': ('bs_grad = 0', {'bs_grad':0}, [('bs_grad = 1',{'bs_grad':1})]),
 'Telecommute (tcom_no)':('tcom_no = 0', {'tcom_no':0}, [('tcom_no = 1',{'tcom_no':1})]),
 'Air conditioning (ac_c)':('ac_c = 0', {'ac_c':0}, [('ac_c = 1',{'ac_c':1})]),
 'Employment Density':  ('Low', {'EmpDens_medium':0,'EmpDens_high':0},
                          [('Medium',{'EmpDens_medium':1,'EmpDens_high':0}),
                           ('High',  {'EmpDens_medium':0,'EmpDens_high':1})]),
 'Land-use Diversity':  ('Not high', {'Diversity_high':0}, [('High',{'Diversity_high':1})]),
 'Network Density':     ('Low', {'NetworkDensity_medium':0,'NetworkDensity_high':0},
                          [('Medium',{'NetworkDensity_medium':1,'NetworkDensity_high':0}),
                           ('High',  {'NetworkDensity_medium':0,'NetworkDensity_high':1})]),
}
INCLUDE_EXTRA = True

CONTINUOUS = [('Community Resilience','CR'),('Social Engagement','SE'),('Personal Resilience','PR')]
CONT_SHIFTS = [('1 Unit Increase*', 1.0)]

def panel_b(ev, act, include_extra=None):
    """Every Panel B contrast for ONE model (event x activity), ALL response categories, long format."""
    include_extra = INCLUDE_EXTRA if include_extra is None else include_extra
    segs = {**SEGMENTS, **(EXTRA_SEGMENTS if include_extra else {})}
    rows = []
    def add(group, base_lab, comp_lab, t, ok):
        for _, r in t.iterrows():
            rows.append(dict(event=ev, event_label=EVENTS[ev]['label'], activity=act,
                             activity_label=ACT_LABEL[act], n=t.attrs['n'],
                             variable_group=group, base=base_lab, comparison=comp_lab,
                             in_model=ok, model_vars=','.join(t.attrs['vars_in_model']),
                             response=r.Response, P_base=r.P_base, P_comp=r.P_comp,
                             ATE_abs=r.ATE_abs, ATE_pct=r.ATE_pct))
    for name, (base_lab, base_spec, comps) in segs.items():
        for comp_lab, comp_spec in comps:
            t, ok = ate_segment(ev, act, base_spec, comp_spec)
            add(name, base_lab, comp_lab, t, ok)
    for name, var in CONTINUOUS:
        for lab, unit_shift in CONT_SHIFTS:
            t, ok = ate_continuous(ev, act, var, unit_shift=unit_shift)
            add(name, 'Current value', lab, t, ok)
    return pd.DataFrame(rows)

def panel_b_wide(long_df, value='ATE_pct'):
    """One row per contrast, one column per response category."""
    resp_order = [r for r in list(CAT3.values()) + list(CAT5.values()) if r in long_df.response.unique()]
    keys = ['variable_group','base','comparison','in_model']
    order = pd.MultiIndex.from_frame(long_df[keys].drop_duplicates())
    w = long_df.set_index(keys + ['response'])[value].unstack('response').reindex(order)[resp_order]
    w.columns = [f'{value} | {c}' for c in w.columns]
    return w.reset_index()

# ------------------------- CHANGE THESE -------------------------
EVENT, ACTIVITY = 'heat', 'Car'
SHOW_ONLY_IN_MODEL = False      # True -> hide contrasts whose variables are not in this model
# ----------------------------------------------------------------

pb = panel_b(EVENT, ACTIVITY)
wide = panel_b_wide(pb, 'ATE_pct').merge(panel_b_wide(pb, 'ATE_abs'),
                                          on=['variable_group','base','comparison','in_model'])
if SHOW_ONLY_IN_MODEL:
    wide = wide[wide.in_model]
print(f'{EVENTS[EVENT]["label"]} — {ACT_LABEL[ACTIVITY]} — N = {pb.n.iloc[0]}, severity held at observed values')
print('='*140)
print(wide.to_string(index=False, float_format=lambda x: f'{x:8.4f}'))

Extreme Heat — Using a car for traveling — N = 2653, severity held at observed values
         variable_group            base        comparison  in_model  ATE_pct | Do less  ATE_pct | About the same  ATE_pct | Do more  ATE_abs | Do less  ATE_abs | About the same  ATE_abs | Do more
                 Gender            Male            Female     False             0.0000                    0.0000             0.0000             0.0000                    0.0000             0.0000
              Age Group           18-30             31-50     False             0.0000                    0.0000             0.0000             0.0000                    0.0000             0.0000
              Age Group           18-30             51-65     False             0.0000                    0.0000             0.0000             0.0000                    0.0000             0.0000
              Age Group           18-30               65+     False             0.0000                    0.0000             0.000

### 5b. Coverage check — is every model variable reachable from Panel B?

Lists, per model, any variable in the specification that no segment (or CR/SE/PR) manipulates.
An empty result means every coefficient in every model shows up in some Panel B contrast.

In [8]:
covered = {v for _,(_,b,_) in {**SEGMENTS, **EXTRA_SEGMENTS}.items() for v in b} | {v for _,v in CONTINUOUS}
gaps = []
for ev in EVENTS:
    for act in available_activities(ev):
        x_vars = COEF[(COEF.event==ev)&(COEF.activity==act)&(COEF.term=='coef')].variable
        miss = [v for v in x_vars if '_imp_' not in v and v not in covered]
        if miss: gaps.append(dict(event=ev, activity=act, not_covered=', '.join(miss)))
print('all model variables covered by Panel B' if not gaps else pd.DataFrame(gaps).to_string(index=False))

all model variables covered by Panel B


## 6. Export

### 6a. Panel 1 (Panel A) — severity

Writes every severity pair for every event and activity to one tidy CSV — 5 events × all activities ×
20 ordered level pairs × each response category. Useful as the data source behind the dashboard.

In [10]:
rows=[]
for ev in EVENTS:
    for act in available_activities(ev):
        for b,c in itertools.permutations([1,2,3,4,5], 2):
            t = ate_severity(ev, act, b, c)
            for _,r in t.iterrows():
                rows.append(dict(event=ev, event_label=EVENTS[ev]['label'], activity=act,
                                 activity_label=ACT_LABEL[act], base_level=b, base_label=SEV_LABEL[b],
                                 comp_level=c, comp_label=SEV_LABEL[c], response=r.Response,
                                 P_base=r.P_base, P_comp=r.P_comp, ATE_abs=r.ATE_abs,
                                 ATE_pct=r.ATE_pct, n=t.attrs['n']))
allpairs = pd.DataFrame(rows)
allpairs.to_csv('./ATE_severity_all_events.csv', index=False)
print(allpairs.shape, '-> ATE_severity_all_events.csv')
print(allpairs.groupby('event_label').size())

(2100, 14) -> ATE_severity_all_events.csv
event_label
Earthquakes     300
Extreme Cold    540
Extreme Heat    540
Flooding        300
Power Outage    420
dtype: int64


### 6b. Panel 2 (Panel B) — population segments, every model × every response

Loops over all 35 models (5 events × their activities) and every response category, using each
model's own specification. Writes:

- `ATE_segments_all_events.csv` — tidy long table, one row per event × activity × contrast × response
  (the dashboard data source).
- `ATE_segments_by_model.xlsx` — one sheet per model (e.g. `heat_Car`), contrasts as rows and
  *Do less / About the same / Do more* as columns for `P_base`, `P_comp`, `ATE_abs` and `ATE_pct`,
  plus an `index` sheet. Needs `openpyxl`; the CSV is written either way.

In [11]:
frames = []
for ev in EVENTS:
    for act in available_activities(ev):
        frames.append(panel_b(ev, act))
seg_all = pd.concat(frames, ignore_index=True)
seg_all.to_csv('./ATE_segments_all_events.csv', index=False)
print(seg_all.shape, '-> ATE_segments_all_events.csv')
print(seg_all.groupby(['event_label','response']).size().unstack(fill_value=0))

try:
    keys = ['variable_group','base','comparison','in_model']
    index_rows = []
    with pd.ExcelWriter('./ATE_segments_by_model.xlsx') as xw:
        for (ev, act), g in seg_all.groupby(['event','activity'], sort=False):
            w = panel_b_wide(g, 'P_base')
            for val in ['P_comp','ATE_abs','ATE_pct']:
                w = w.merge(panel_b_wide(g, val), on=keys)
            sheet = f'{ev}_{act}'.replace(' ', '_')[:31]
            w.to_excel(xw, sheet_name=sheet, index=False)
            index_rows.append(dict(sheet=sheet, event=EVENTS[ev]['label'], activity=ACT_LABEL[act],
                                   n=int(g.n.iloc[0]), contrasts=len(w),
                                   contrasts_in_model=int(w.in_model.sum())))
        pd.DataFrame(index_rows).to_excel(xw, sheet_name='index', index=False)
    print(f'{len(index_rows)} model sheets -> ATE_segments_by_model.xlsx')
except ImportError:
    print('openpyxl not installed - skipped the Excel workbook (pip install openpyxl). CSV was written.')

(3990, 15) -> ATE_segments_all_events.csv
response      About the same  Do less  Do more
event_label                                   
Earthquakes              190      190      190
Extreme Cold             342      342      342
Extreme Heat             342      342      342
Flooding                 190      190      190
Power Outage             266      266      266
35 model sheets -> ATE_segments_by_model.xlsx
